# STO-MILP v10 — BH Engineering Spec Final v2.3

**Key changes from v10:**
- PV is exogenous (no `cap_pv`); Stage 1 sizes only CC, P_B, E_B
- Spec §6.2 TOU_FixedPeak tariff (summer/non-summer × weekday/Saturday/Sunday)
- Summer/non-summer basic charge (223.6 / 166.9 NTD/kW-month)
- C14 PWL degradation with spec breakpoints b_k, μ_k, λ_k (replaces flat rate)
- Green SOC inter/intra-day superposition matching Method 1 structure
- Green SOC init E_g,inter_1 = 0 and terminal constraint C13f
- Scenario-wise D_max_{m,ω} for monthly billing

In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import time
import json
from pathlib import Path

from milp_common import get_config, load_data, CASE_TABLE, format_results, get_monthly_basic_charge

CFG = get_config()
print(f"\nOutput dir: {CFG['output_dir']}")

CRF PV (20yr, r=0.05): 0.0802
CRF BESS (15yr, r=0.05): 0.0963

Output dir: ../milp_outputs


## Model Builder (Spec v2.3)

Two-stage stochastic MILP: Stage 1 sizes CC, P_B, E_B. Stage 2 dispatches per repday×scenario×hour.

Objective: Min AEC_total = AEC_inv + AEC_ene + AEC_basic + AEC_over + AEC_green + AEC_deg

In [2]:
def build_and_solve(CFG, repday_data, calendar_order, info, case_flags):
    """Build and solve the v10 MILP per spec v2.3.

    Returns: dict with capacities, objective, RE%, T-REC cost, solve time, cost breakdown.
    """
    use_method1 = case_flags.get('method1', True) and calendar_order is not None
    n_repdays = info['n_repdays']
    n_scenarios = info['n_scenarios']
    n_hours = info['n_hours']
    N = n_repdays * n_scenarios * n_hours
    n_seg = len(CFG['pwl_deg_lambda_k'])  # 4 PWL segments
    PV_RATING = CFG['pv_rating_kw']  # 50 kW reference system

    # ── Pre-compute flat arrays ──
    pv_coeff  = np.empty(N)    # PV coefficient: pv_kw / PV_RATING (multiply by cap_pv)
    load_arr  = np.empty(N)
    pw_arr    = np.empty(N)    # prob × weight
    tou_arr   = np.empty(N)
    month_arr = np.empty(N, dtype=int)

    def idx(d, s, t):
        return d * (n_scenarios * n_hours) + s * n_hours + t

    for d in range(n_repdays):
        dd = repday_data[d]
        for s in range(n_scenarios):
            sc = dd['scenarios'][s]
            start = idx(d, s, 0)
            sl = slice(start, start + n_hours)
            pv_coeff[sl] = sc['pv_kw'] / PV_RATING   # normalized coefficient
            load_arr[sl] = sc['load_kw']
            pw_arr[sl] = sc['prob'] * dd['weight']
            tou_arr[sl] = dd['tou']
            month_arr[sl] = dd['month']

    # First-hour indices for SOC reset (per repday-scenario)
    is_first = np.zeros(N, dtype=bool)
    for d in range(n_repdays):
        for s in range(n_scenarios):
            is_first[idx(d, s, 0)] = True
    first_hours = np.where(is_first)[0]
    later_hours = np.where(~is_first)[0]
    prev_idx_arr = np.arange(N) - 1

    # ── Build Gurobi model ──
    m = gp.Model('milp_v10')
    m.Params.OutputFlag = 1
    m.Params.TimeLimit = CFG['time_limit']
    m.Params.Method = 2
    m.Params.Presolve = 2
    m.Params.Cuts = 2
    m.Params.MIPGap = 1e-4
    m.Params.MIPFocus = 1

    # ── Stage 1: Capacity variables (spec §7) ──
    cap_pv = m.addVar(ub=CFG['pv_max_kw'], name='cap_pv')         # PV capacity (kW)
    cap_bp = m.addVar(ub=CFG['bess_p_max_kw'], name='P_B')       # battery power
    cap_be = m.addVar(ub=CFG['bess_e_max_kwh'], name='E_B')      # battery energy
    cap_cd = m.addVar(name='CC')                                   # contract capacity

    # ── Stage 2: Dispatch variables ──
    p_grid    = m.addMVar(N, name='P_grid')
    p_pv_load = m.addMVar(N, name='P_pv_load')    # PV → load
    p_pv_ch   = m.addMVar(N, name='P_pv_ch')      # PV → BESS charge
    p_curt    = m.addMVar(N, name='P_curt')
    p_ch      = m.addMVar(N, name='P_ch')          # total BESS charge
    p_dis     = m.addMVar(N, name='P_dis')         # BESS discharge
    u_bin     = m.addMVar(N, vtype=GRB.BINARY, name='u')  # C4 mutual exclusivity

    # Green SOC variables (spec §7 RE/Green + Chronology/Green)
    p_ch_g    = m.addMVar(N, name='P_ch_g')        # green charging power
    p_dis_g   = m.addMVar(N, name='P_dis_g')       # green discharging power

    # Intra-day SOC displacement (spec C5)
    delta_e   = m.addMVar(N, lb=-GRB.INFINITY, name='dE')

    # PWL degradation segment variables (spec C14a)
    e_dis_seg = m.addMVar((N, n_seg), name='e_dis_seg')

    # T-REC (spec §7 Derived/RE)
    trec_buy  = m.addVar(name='E_TREC')

    m.update()

    eff_ch = CFG['eff_charge']
    eff_dis = CFG['eff_discharge']
    eff_dis_inv = 1.0 / eff_dis
    b_k = CFG['pwl_deg_b_k']
    lam_k = CFG['pwl_deg_lambda_k']

    # ── C1: Power balance (spec §9, no export) ──
    m.addConstrs(
        (p_grid[i] + p_pv_load[i] + p_dis[i] == load_arr[i] + p_ch[i]
         for i in range(N)), name='C1')

    # ── C2: PV availability split (spec §9) ──
    # PV output = pv_coeff[i] * cap_pv (scales bridge 50kW reference to chosen capacity)
    m.addConstrs(
        (p_pv_load[i] + p_pv_ch[i] + p_curt[i] == pv_coeff[i] * cap_pv
         for i in range(N)), name='C2')
    # PV_to_charge is a subset of total charge
    m.addConstrs((p_pv_ch[i] <= p_ch[i] for i in range(N)), name='C2b')

    # ── C3: Charge/discharge upper limits (spec §9) ──
    # ── C4: Mutual exclusivity (spec §9, retained in baseline) ──
    m.addConstrs((p_ch[i]  <= cap_bp * u_bin[i]       for i in range(N)), name='C3_ch')
    m.addConstrs((p_dis[i] <= cap_bp * (1 - u_bin[i]) for i in range(N)), name='C3_dis')

    # ── C5: Intra-day SOC displacement update (spec §9) ──
    m.addConstrs(
        (delta_e[i] == eff_ch * p_ch[i] - eff_dis_inv * p_dis[i]
         for i in first_hours), name='C5_init')
    m.addConstrs(
        (delta_e[i] == delta_e[prev_idx_arr[i]] + eff_ch * p_ch[i] - eff_dis_inv * p_dis[i]
         for i in later_hours), name='C5_cont')

    # ── C6-C9: SOC bounds and inter-day linkage ──
    if use_method1:
        n_cal = len(calendar_order)
        E_inter = m.addMVar(n_cal, lb=0, name='E_inter')

        # C8: SOC bounds via superposition
        for cal_idx, cal_day in enumerate(calendar_order):
            d = cal_day['d_idx']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    flat = idx(d, s, t)
                    m.addConstr(E_inter[cal_idx] + delta_e[flat] >= CFG['soc_min'] * cap_be,
                                name=f'C8lo_{cal_idx}_{s}_{t}')
                    m.addConstr(E_inter[cal_idx] + delta_e[flat] <= CFG['soc_max'] * cap_be,
                                name=f'C8hi_{cal_idx}_{s}_{t}')

        # C7: Inter-day update (expectation-based)
        for cal_idx in range(n_cal - 1):
            d = calendar_order[cal_idx]['d_idx']
            expected_de = gp.LinExpr()
            for s in range(n_scenarios):
                prob_s = repday_data[d]['scenarios'][s]['prob']
                last_idx = idx(d, s, n_hours - 1)
                expected_de += prob_s * delta_e[last_idx]
            m.addConstr(E_inter[cal_idx + 1] == E_inter[cal_idx] + expected_de,
                        name=f'C7_{cal_idx}')

        # C9: Initial/terminal conditions
        m.addConstr(E_inter[0] == CFG['soc_init'] * cap_be, name='C9_init')

        d_last = calendar_order[-1]['d_idx']
        expected_de_last = gp.LinExpr()
        for s in range(n_scenarios):
            prob_s = repday_data[d_last]['scenarios'][s]['prob']
            last_idx = idx(d_last, s, n_hours - 1)
            expected_de_last += prob_s * delta_e[last_idx]
        E_end = E_inter[n_cal - 1] + expected_de_last
        m.addConstr(E_end - E_inter[0] <=  CFG['epsilon_term'] * cap_be, name='C9_term_hi')
        m.addConstr(E_end - E_inter[0] >= -CFG['epsilon_term'] * cap_be, name='C9_term_lo')

        # ── C13: Green SOC with inter/intra superposition ──
        E_g_inter = m.addMVar(n_cal, lb=0, name='E_g_inter')
        delta_e_g = m.addMVar(N, lb=-GRB.INFINITY, name='dE_g')

        m.addConstrs((p_ch_g[i] <= p_pv_ch[i] for i in range(N)), name='C13_ch_g')
        m.addConstrs((p_dis_g[i] <= p_dis[i] for i in range(N)), name='C13_dis_g')

        m.addConstrs(
            (delta_e_g[i] == eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i]
             for i in first_hours), name='C13a_init')
        m.addConstrs(
            (delta_e_g[i] == delta_e_g[prev_idx_arr[i]] + eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i]
             for i in later_hours), name='C13a_cont')

        for cal_idx, cal_day in enumerate(calendar_order):
            d = cal_day['d_idx']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    flat = idx(d, s, t)
                    m.addConstr(E_g_inter[cal_idx] + delta_e_g[flat] >= 0,
                                name=f'C13d_lo_{cal_idx}_{s}_{t}')
                    m.addConstr(
                        E_g_inter[cal_idx] + delta_e_g[flat] <=
                        E_inter[cal_idx] + delta_e[flat],
                        name=f'C13d_hi_{cal_idx}_{s}_{t}')

        for cal_idx in range(n_cal - 1):
            d = calendar_order[cal_idx]['d_idx']
            expected_de_g = gp.LinExpr()
            for s in range(n_scenarios):
                prob_s = repday_data[d]['scenarios'][s]['prob']
                last_idx = idx(d, s, n_hours - 1)
                expected_de_g += prob_s * delta_e_g[last_idx]
            m.addConstr(E_g_inter[cal_idx + 1] == E_g_inter[cal_idx] + expected_de_g,
                        name=f'C13c_{cal_idx}')

        m.addConstr(E_g_inter[0] == 0, name='C13e')

        d_last = calendar_order[-1]['d_idx']
        expected_de_g_last = gp.LinExpr()
        for s in range(n_scenarios):
            prob_s = repday_data[d_last]['scenarios'][s]['prob']
            last_idx = idx(d_last, s, n_hours - 1)
            expected_de_g_last += prob_s * delta_e_g[last_idx]
        E_g_end = E_g_inter[n_cal - 1] + expected_de_g_last
        m.addConstr(E_g_end - E_g_inter[0] <=  CFG['epsilon_g_term'] * cap_be, name='C13f_hi')
        m.addConstr(E_g_end - E_g_inter[0] >= -CFG['epsilon_g_term'] * cap_be, name='C13f_lo')

    else:
        # No Method 1: daily SOC reset
        for i in range(N):
            m.addConstr(CFG['soc_init'] * cap_be + delta_e[i] >= CFG['soc_min'] * cap_be,
                        name=f'C8lo_{i}')
            m.addConstr(CFG['soc_init'] * cap_be + delta_e[i] <= CFG['soc_max'] * cap_be,
                        name=f'C8hi_{i}')

        delta_e_g = m.addMVar(N, lb=-GRB.INFINITY, name='dE_g')
        m.addConstrs((p_ch_g[i] <= p_pv_ch[i] for i in range(N)), name='C13_ch_g')
        m.addConstrs((p_dis_g[i] <= p_dis[i] for i in range(N)), name='C13_dis_g')
        m.addConstrs(
            (delta_e_g[i] == eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i]
             for i in first_hours), name='C13a_init')
        m.addConstrs(
            (delta_e_g[i] == delta_e_g[prev_idx_arr[i]] + eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i]
             for i in later_hours), name='C13a_cont')
        for i in range(N):
            m.addConstr(delta_e_g[i] >= 0, name=f'C13d_lo_{i}')
            m.addConstr(delta_e_g[i] <= CFG['soc_init'] * cap_be + delta_e[i],
                        name=f'C13d_hi_{i}')

    # ── C10: Monthly maximum-demand proxy (spec §9) ──
    # Use all 12 months to avoid KeyError when calendar_order references months not in repday data
    all_months = list(range(1, 13))

    if use_method1:
        D_max = {mo: m.addVar(name=f'Dmax_{mo}') for mo in all_months}
        kappa = CFG['kappa']
        for cal_idx, cal_day in enumerate(calendar_order):
            d = cal_day['d_idx']
            mo = cal_day['month_id']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    flat = idx(d, s, t)
                    m.addConstr(D_max[mo] >= kappa * p_grid[flat],
                                name=f'C10_{mo}_{cal_idx}_{s}_{t}')
    else:
        months_in_data = sorted(set(month_arr))
        D_max = {mo: m.addVar(name=f'Dmax_{mo}') for mo in months_in_data}
        all_months = months_in_data  # only use months that appear
        kappa = CFG['kappa']
        for i in range(N):
            mo = int(month_arr[i])
            m.addConstr(D_max[mo] >= kappa * p_grid[i], name=f'C10_{mo}_{i}')

    # ── C11: Piecewise over-contract (spec §9) ──
    oc_seg1 = {mo: m.addVar(name=f'O1_{mo}') for mo in all_months}
    oc_seg2 = {mo: m.addVar(name=f'O2_{mo}') for mo in all_months}
    for mo in all_months:
        m.addConstr(oc_seg1[mo] >= D_max[mo] - cap_cd, name=f'C11a_{mo}')
        m.addConstr(oc_seg1[mo] <= 0.10 * cap_cd, name=f'C11b_{mo}')
        m.addConstr(oc_seg2[mo] >= D_max[mo] - 1.10 * cap_cd, name=f'C11c_{mo}')

    # ── C12: RE20 + T-REC (spec §9) ──
    total_load = float(np.sum(pw_arr * load_arr))

    re_expr = gp.quicksum(
        pw_arr[i] * (p_pv_load[i] + p_dis_g[i]) for i in range(N))
    m.addConstr(re_expr + trec_buy >= CFG['re_target'] * total_load, name='C12_RE20')

    # ── C14: PWL degradation (spec §9, mandatory hard spec) ──
    for i in range(N):
        m.addConstr(
            gp.quicksum(e_dis_seg[i, k] for k in range(n_seg)) == p_dis[i],
            name=f'C14a_{i}')
        for k in range(n_seg):
            seg_width = b_k[k + 1] - b_k[k]
            m.addConstr(e_dis_seg[i, k] <= seg_width * cap_be,
                        name=f'C14b_{i}_{k}')

    # ── Objective: Min AEC_total (spec §8) ──

    # AEC_inv: PV annuity + BESS annuity + FOM
    AEC_inv = (
        cap_pv * CFG['capex_pv_per_kw'] * CFG['crf_pv']
        + cap_bp * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
        + cap_be * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess']
                    + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate'])
    )

    # AEC_ene: grid energy purchase (weighted)
    AEC_ene = gp.quicksum(
        pw_arr[i] * p_grid[i] * float(tou_arr[i]) for i in range(N))

    # AEC_basic: contract demand basic charge (summer/non-summer)
    AEC_basic = gp.LinExpr()
    for mo in all_months:
        rate = get_monthly_basic_charge(mo, CFG)
        AEC_basic += cap_cd * rate

    # AEC_over: over-contract penalty (summer/non-summer rates)
    AEC_over = gp.LinExpr()
    for mo in all_months:
        rate = get_monthly_basic_charge(mo, CFG)
        AEC_over += (oc_seg1[mo] * rate * CFG['oc_within_10pct_mult']
                     + oc_seg2[mo] * rate * CFG['oc_beyond_10pct_mult'])

    # AEC_green: T-REC top-up cost
    AEC_green = CFG['trec_cost_per_kwh'] * trec_buy

    # AEC_deg: PWL degradation cost (spec C14c)
    if use_method1:
        n_d = {}
        for cal_day in calendar_order:
            d = cal_day['d_idx']
            n_d[d] = n_d.get(d, 0) + 1
        AEC_deg = gp.LinExpr()
        for d in range(n_repdays):
            mult = n_d.get(d, repday_data[d]['weight'])
            for s in range(n_scenarios):
                prob_s = repday_data[d]['scenarios'][s]['prob']
                for t in range(n_hours):
                    flat = idx(d, s, t)
                    for k in range(n_seg):
                        AEC_deg += mult * prob_s * lam_k[k] * e_dis_seg[flat, k]
    else:
        AEC_deg = gp.LinExpr()
        for i in range(N):
            for k in range(n_seg):
                AEC_deg += pw_arr[i] * lam_k[k] * e_dis_seg[i, k]

    m.setObjective(AEC_inv + AEC_ene + AEC_basic + AEC_over + AEC_green + AEC_deg,
                   GRB.MINIMIZE)

    # ── Solve ──
    print(f'  Variables: {m.NumVars:,} ({m.NumIntVars:,} binary)')
    print(f'  Constraints: {m.NumConstrs:,}')
    t0 = time.time()
    m.optimize()
    solve_time = time.time() - t0

    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        print(f'  WARNING: Solver status = {m.Status}')
        return None

    # ── Extract results ──
    pv_val = cap_pv.X
    bp_val = cap_bp.X
    be_val = cap_be.X
    cd_val = cap_cd.X
    obj = m.ObjVal

    # RE percentage
    re_val = sum(pw_arr[i] * (p_pv_load[i].X + p_dis_g[i].X) for i in range(N))
    re_pct = re_val / total_load * 100
    trec_val = trec_buy.X * CFG['trec_cost_per_kwh']

    # Cost breakdown
    cost_breakdown = {
        'AEC_inv': AEC_inv.getValue(),
        'AEC_ene': AEC_ene.getValue(),
        'AEC_basic': AEC_basic.getValue(),
        'AEC_over': AEC_over.getValue(),
        'AEC_green': AEC_green.getValue(),
        'AEC_deg': AEC_deg.getValue(),
    }

    # Battery throughput and equivalent cycles
    total_discharge = sum(pw_arr[i] * p_dis[i].X for i in range(N))
    equiv_cycles = total_discharge / max(be_val, 1.0) if be_val > 0 else 0

    print(f'  Objective: {obj:,.0f} TWD ({obj/1e6:.2f}M)')
    print(f'  PV={pv_val:,.0f} kW, BESS={bp_val:,.0f}/{be_val:,.0f} kW/kWh, Contract={cd_val:,.0f} kW')
    print(f'  RE={re_pct:.1f}%, T-REC={trec_buy.X:,.0f} kWh ({trec_val/1e6:.2f}M TWD)')
    print(f'  Degradation: {cost_breakdown["AEC_deg"]/1e6:.2f}M, '
          f'Throughput: {total_discharge:,.0f} kWh, Equiv cycles: {equiv_cycles:.0f}')
    print(f'  Solve: {solve_time:.1f}s, Gap: {m.MIPGap*100:.4f}%')

    return {
        'cap_pv': pv_val, 'cap_bp': bp_val, 'cap_be': be_val, 'cap_cd': cd_val,
        'obj': obj, 're_pct': re_pct, 'trec_cost': trec_val,
        'solve_time': solve_time, 'gap': m.MIPGap,
        'cost_breakdown': cost_breakdown,
        'total_discharge': total_discharge, 'equiv_cycles': equiv_cycles,
    }

## Run All 8 Cases

In [3]:
all_results = []

for case in CASE_TABLE:
    name = case['name']
    print(f"\n{'='*60}")
    print(f"  CASE: {name}")
    print(f"{'='*60}")
    print(f"  Method1={case['method1']}, RiskDays={case['risk_days']}, "
          f"ProbPV={case['prob_pv']}, Uplift={case['uplift']}")

    repday_data, calendar_order, info = load_data(CFG, case)
    result = build_and_solve(CFG, repday_data, calendar_order, info, case)

    if result is not None:
        formatted = format_results(
            name, result['cap_pv'], result['cap_bp'], result['cap_be'], result['cap_cd'],
            result['obj'], result['re_pct'], result['trec_cost'],
            result['solve_time'], result['cost_breakdown'], CFG)
        formatted['equiv_cycles'] = round(result['equiv_cycles'], 0)
        formatted['discharge_MWh'] = round(result['total_discharge'] / 1e3, 1)
        all_results.append(formatted)
    else:
        all_results.append({'case': name, 'status': 'infeasible'})

print(f"\n\n{'='*60}")
print(f"  ALL {len(all_results)} CASES COMPLETE")
print(f"{'='*60}")


  CASE: M0_I0_R0
  Method1=False, RiskDays=False, ProbPV=False, Uplift=None
  Repdays: 16 (16 body + 0 risk)
  Scenarios: 1/repday, 24 hours
  Weight sum: 365
  Calendar days: N/A (no Method 1)
Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2797924


Academic license 2797924 - for non-commercial use only - registered to m1___@mail.ntust.edu.tw


Set parameter OutputFlag to value 1


...

Solution count 1: 9.6527e+07 


Optimal solution found (tolerance 1.00e-04)


Best objective 9.652704015246e+07, best bound 9.652704015246e+07, gap 0.0000%


  Objective: 96,527,040 TWD (96.53M)
  PV=7,817 kW, BESS=1,326/8,002 kW/kWh, Contract=3,049 kW
  RE=40.3%, T-REC=0 kWh (0.00M TWD)
  Degradation: 2.46M, Throughput: 2,359,641 kWh, Equiv cycles: 295
  Solve: 20.4s, Gap: 0.0000%



  ALL 8 CASES COMPLETE


## Results Comparison Table

In [4]:
results_df = pd.DataFrame(all_results)

display_cols = ['case', 'pv_kw', 'bess_p_kw', 'bess_e_kwh', 'ep_ratio', 'contract_kw',
                'total_cost_M', 'AEC_inv_M', 'AEC_ene_M', 'AEC_basic_M',
                'AEC_over_M', 'AEC_green_M', 'AEC_deg_M',
                're_pct', 'equiv_cycles', 'discharge_MWh', 'solve_s']
cols = [c for c in display_cols if c in results_df.columns]
print(results_df[cols].to_string(index=False))

# Save
out_path = Path(CFG['output_dir'])
results_df.to_csv(out_path / 'case_summary_main.csv', index=False)
with open(out_path / 'case_results_all.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"\nResults saved to {out_path}")

       case  pv_kw  bess_p_kw  bess_e_kwh  ep_ratio  contract_kw  total_cost_M  AEC_inv_M  AEC_ene_M  AEC_basic_M  AEC_over_M  AEC_green_M  AEC_deg_M  re_pct  equiv_cycles  discharge_MWh  solve_s
   M0_I0_R0 7467.2     1709.9     12565.9       7.3       2701.4         84.69      36.28      39.36         4.67        0.11          0.0       4.27    40.6         332.0         4173.5      0.4
   M1_I0_R0 7197.0     1116.3      4652.1       4.2       2786.6         90.18      28.21      53.95         6.53        0.00          0.0       1.49    36.7         321.0         1495.2      3.8
   M2_I0_R0 7719.3     1447.0      7282.7       5.0       2938.4         94.10      32.43      52.15         6.88        0.28          0.0       2.36    40.2         307.0         2239.1     17.5
   M2_I1_R0 7466.1     1362.5      7623.9       5.6       3015.5         94.92      31.81      53.33         7.07        0.31          0.0       2.41    39.4         303.0         2307.6     21.6
M2_I1_R1_p3 7692.3  

## Key Comparisons

In [5]:
if len(results_df) >= 4 and 'total_cost_M' in results_df.columns:
    # Value of probabilistic info
    r_i0 = results_df[results_df['case'] == 'M2_I0_R0']
    r_i1 = results_df[results_df['case'] == 'M2_I1_R0']
    if len(r_i0) > 0 and len(r_i1) > 0:
        cost_diff = r_i0['total_cost_M'].iloc[0] - r_i1['total_cost_M'].iloc[0]
        print(f"Value of probabilistic PV info: {cost_diff:.2f}M TWD")
        print(f"  M2_I0_R0 (deterministic): {r_i0['total_cost_M'].iloc[0]:.2f}M")
        print(f"  M2_I1_R0 (probabilistic): {r_i1['total_cost_M'].iloc[0]:.2f}M")

    # Method 1 progression
    print(f"\nMethod skeleton progression:")
    for c in ['M0_I0_R0', 'M1_I0_R0', 'M2_I0_R0']:
        row = results_df[results_df['case'] == c]
        if len(row) > 0:
            print(f"  {c}: {row['total_cost_M'].iloc[0]:.2f}M TWD, "
                  f"BESS={row['bess_p_kw'].iloc[0]:.0f}/{row['bess_e_kwh'].iloc[0]:.0f}, "
                  f"CC={row['contract_kw'].iloc[0]:.0f}")

    # Robustness
    print(f"\nRobustness (load uplift):")
    for c in ['M2_I1_R0', 'M2_I1_R1_p3', 'M2_I1_R1_p5', 'M2_I1_R2_p3', 'M2_I1_R2_p5']:
        row = results_df[results_df['case'] == c]
        if len(row) > 0:
            print(f"  {c}: {row['total_cost_M'].iloc[0]:.2f}M, "
                  f"CC={row['contract_kw'].iloc[0]:.0f} kW, "
                  f"RE={row['re_pct'].iloc[0]:.1f}%, "
                  f"AEC_deg={row['AEC_deg_M'].iloc[0]:.2f}M")

    # Cost decomposition for mainline
    print(f"\nCost decomposition (M2_I1_R0 mainline):")
    main = results_df[results_df['case'] == 'M2_I1_R0']
    if len(main) > 0:
        for col in ['AEC_inv_M', 'AEC_ene_M', 'AEC_basic_M', 'AEC_over_M', 'AEC_green_M', 'AEC_deg_M']:
            if col in main.columns:
                print(f"  {col}: {main[col].iloc[0]:.2f}M")
else:
    print('Not enough completed cases for comparison')

Value of probabilistic PV info: -0.82M TWD
  M2_I0_R0 (deterministic): 94.10M
  M2_I1_R0 (probabilistic): 94.92M

Method skeleton progression:
  M0_I0_R0: 84.69M TWD, BESS=1710/12566, CC=2701
  M1_I0_R0: 90.18M TWD, BESS=1116/4652, CC=2787
  M2_I0_R0: 94.10M TWD, BESS=1447/7283, CC=2938

Robustness (load uplift):
  M2_I1_R0: 94.92M, CC=3016 kW, RE=39.4%, AEC_deg=2.41M
  M2_I1_R1_p3: 97.77M, CC=3105 kW, RE=38.8%, AEC_deg=2.48M
  M2_I1_R1_p5: 99.67M, CC=3165 kW, RE=39.4%, AEC_deg=2.53M
  M2_I1_R2_p3: 95.88M, CC=3031 kW, RE=39.7%, AEC_deg=2.45M
  M2_I1_R2_p5: 96.53M, CC=3049 kW, RE=40.3%, AEC_deg=2.46M

Cost decomposition (M2_I1_R0 mainline):
  AEC_inv_M: 31.81M
  AEC_ene_M: 53.33M
  AEC_basic_M: 7.07M
  AEC_over_M: 0.31M
  AEC_green_M: 0.00M
  AEC_deg_M: 2.41M


## Replay / Audit (Spec §11)

Fix sizing (CC*, P_B*, E_B*) from the mainline case and replay through the full-year daily package.
Outputs: replay annual cost, monthly max demand, over-contract months, RE20/T-REC, design-to-replay gap.

In [6]:
def replay_audit(CFG, sizing, full_year_df):
    """Fixed-design full-year replay/audit (spec §11).

    Args:
        sizing: dict with CC, P_B, E_B, cap_pv from MILP solution
        full_year_df: DataFrame with columns [date, hour, load_kw, pv_kw, month, dow, day]
                      One row per hour for 365 days (8760 rows).
    Returns:
        dict with replay results.
    """
    CC = sizing['CC']
    P_B = sizing['P_B']
    E_B = sizing['E_B']
    cap_pv_val = sizing.get('cap_pv', 0)

    eff_ch = CFG['eff_charge']
    eff_dis = CFG['eff_discharge']
    soc_min_kwh = CFG['soc_min'] * E_B
    soc_max_kwh = CFG['soc_max'] * E_B
    kappa = CFG['kappa']
    b_k = CFG['pwl_deg_b_k']
    lam_k = CFG['pwl_deg_lambda_k']

    # State
    soc = CFG['soc_init'] * E_B
    green_soc = 0.0  # E_g,inter_1 = 0

    # Accumulators
    monthly_results = {}
    annual_grid_cost = 0.0
    annual_deg_cost = 0.0
    annual_pv_self = 0.0
    annual_green_dis = 0.0
    annual_load = 0.0
    annual_discharge = 0.0

    for _, row in full_year_df.iterrows():
        load = row['load_kw']
        pv = row['pv_kw']
        month = int(row['month'])
        dow = int(row['dow'])
        day = int(row['day'])
        hour = int(row['hour'])

        tou = get_tou_price(month, day, dow, hour)

        # Greedy dispatch: PV→load first, excess→charge, deficit→discharge/grid
        pv_to_load = min(pv, load)
        pv_excess = pv - pv_to_load
        load_remaining = load - pv_to_load

        # Charge from PV excess
        charge_room = min(P_B, (soc_max_kwh - soc) / eff_ch)
        pv_to_charge = min(pv_excess, charge_room)
        pv_curtailed = pv_excess - pv_to_charge

        # Discharge to meet remaining load
        discharge_room = min(P_B, (soc - soc_min_kwh) * eff_dis)
        discharge = min(load_remaining, discharge_room)
        grid_import = load_remaining - discharge

        # Update SOC
        soc += eff_ch * pv_to_charge - (1.0 / eff_dis) * discharge

        # Green SOC tracking
        green_charge = pv_to_charge  # all PV charge is green
        green_discharge = min(discharge, green_soc * eff_dis) if green_soc > 0 else 0.0
        green_soc += eff_ch * green_charge - (1.0 / eff_dis) * green_discharge
        green_soc = max(0.0, min(green_soc, soc))

        # PWL degradation cost for this hour's discharge
        e_dis = discharge
        deg_cost_h = 0.0
        remaining = e_dis
        for k in range(len(lam_k)):
            seg_cap = (b_k[k + 1] - b_k[k]) * E_B
            seg_used = min(remaining, seg_cap)
            deg_cost_h += lam_k[k] * seg_used
            remaining -= seg_used
            if remaining <= 0:
                break

        # Accumulate monthly
        if month not in monthly_results:
            monthly_results[month] = {
                'grid_cost': 0.0, 'max_demand': 0.0,
                'load': 0.0, 'grid_import': 0.0,
                'pv_self': 0.0, 'green_dis': 0.0, 'deg_cost': 0.0,
            }
        mr = monthly_results[month]
        mr['grid_cost'] += grid_import * tou
        mr['max_demand'] = max(mr['max_demand'], kappa * grid_import)
        mr['load'] += load
        mr['grid_import'] += grid_import
        mr['pv_self'] += pv_to_load
        mr['green_dis'] += green_discharge
        mr['deg_cost'] += deg_cost_h

        annual_grid_cost += grid_import * tou
        annual_deg_cost += deg_cost_h
        annual_pv_self += pv_to_load
        annual_green_dis += green_discharge
        annual_load += load
        annual_discharge += discharge

    # Basic charge and over-contract
    annual_basic = 0.0
    annual_over = 0.0
    over_contract_months = 0
    monthly_bills = {}

    for mo, mr in sorted(monthly_results.items()):
        rate = get_monthly_basic_charge(mo, CFG)
        basic = CC * rate
        over_10 = min(max(mr['max_demand'] - CC, 0), 0.10 * CC)
        over_gt10 = max(mr['max_demand'] - 1.10 * CC, 0)
        penalty = over_10 * rate * CFG['oc_within_10pct_mult'] + over_gt10 * rate * CFG['oc_beyond_10pct_mult']

        if mr['max_demand'] > CC:
            over_contract_months += 1

        annual_basic += basic
        annual_over += penalty
        monthly_bills[mo] = {
            'basic': basic, 'energy': mr['grid_cost'], 'penalty': penalty,
            'total': basic + mr['grid_cost'] + penalty + mr['deg_cost'],
            'max_demand': mr['max_demand'], 'over_contract': mr['max_demand'] > CC,
        }

    # RE20 accounting
    re_total = annual_pv_self + annual_green_dis
    re_pct = re_total / annual_load * 100 if annual_load > 0 else 0
    trec_gap = max(CFG['re_target'] * annual_load - re_total, 0)
    trec_cost = trec_gap * CFG['trec_cost_per_kwh']

    # Investment annuity: PV + BESS
    aec_inv = (cap_pv_val * CFG['capex_pv_per_kw'] * CFG['crf_pv']
               + P_B * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
               + E_B * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess']
                        + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate']))

    replay_total = aec_inv + annual_grid_cost + annual_basic + annual_over + trec_cost + annual_deg_cost
    equiv_cycles = annual_discharge / E_B if E_B > 0 else 0

    monthly_bill_values = [mb['total'] for mb in monthly_bills.values()]

    return {
        'replay_total': replay_total,
        'AEC_inv': aec_inv,
        'AEC_ene': annual_grid_cost,
        'AEC_basic': annual_basic,
        'AEC_over': annual_over,
        'AEC_green': trec_cost,
        'AEC_deg': annual_deg_cost,
        're_pct': re_pct,
        'trec_kWh': trec_gap,
        'trec_cost': trec_cost,
        'over_contract_months': over_contract_months,
        'worst_month_bill': max(monthly_bill_values) if monthly_bill_values else 0,
        'p95_monthly_bill': np.percentile(monthly_bill_values, 95) if monthly_bill_values else 0,
        'equiv_cycles': equiv_cycles,
        'total_discharge': annual_discharge,
        'monthly_bills': monthly_bills,
    }

In [7]:
# Load full-year data for replay
from milp_common import get_tou_price, get_monthly_basic_charge

raw = pd.read_csv(CFG['load_csv']).dropna(subset=['Date'])
raw['ts'] = pd.to_datetime(raw['Date'] + ' ' + raw['Time'])
raw['hour'] = raw['ts'].dt.hour
raw['month'] = raw['ts'].dt.month
raw['day'] = raw['ts'].dt.day
raw['dow'] = raw['ts'].dt.dayofweek
raw['date'] = raw['ts'].dt.date

full_year = raw[['date', 'hour', 'month', 'day', 'dow']].copy()
full_year['load_kw'] = raw['Load_kWh'].values   # hourly load (kWh = kW for 1h)
full_year['pv_kw'] = raw['Solar_kWh'].values     # hourly PV (kWh = kW for 1h)
full_year = full_year.sort_values(['date', 'hour']).reset_index(drop=True)

print(f"Full-year data: {len(full_year)} hours, {full_year['date'].nunique()} days")
print(f"Load range: {full_year['load_kw'].min():.0f} – {full_year['load_kw'].max():.0f} kW")
print(f"PV range: {full_year['pv_kw'].min():.1f} – {full_year['pv_kw'].max():.1f} kW")

Full-year data: 8760 hours, 366 days
Load range: 0 – 5179 kW
PV range: 0.0 – 357.0 kW


In [8]:
# Run replay on mainline and all completed cases
mainline_name = 'M2_I1_R0'
replay_results = {}

for res in all_results:
    if 'status' in res and res['status'] == 'infeasible':
        continue
    case_name = res['case']
    sizing = {
        'CC': res['contract_kw'],
        'P_B': res['bess_p_kw'],
        'E_B': res['bess_e_kwh'],
        'cap_pv': res['pv_kw'],
    }
    print(f"\n--- Replay: {case_name} (PV={sizing['cap_pv']:.0f}, CC={sizing['CC']:.0f}, P_B={sizing['P_B']:.0f}, E_B={sizing['E_B']:.0f}) ---")
    rr = replay_audit(CFG, sizing, full_year)
    replay_results[case_name] = rr

    milp_cost = res['total_cost_M'] * 1e6
    gap = (rr['replay_total'] - milp_cost) / rr['replay_total'] * 100
    print(f"  Replay total: {rr['replay_total']/1e6:.2f}M TWD")
    print(f"  MILP reduced: {milp_cost/1e6:.2f}M TWD")
    print(f"  Design-to-replay gap: {gap:.1f}%")
    print(f"  Over-contract months: {rr['over_contract_months']}")
    print(f"  RE: {rr['re_pct']:.1f}%, T-REC: {rr['trec_kWh']:,.0f} kWh ({rr['trec_cost']/1e6:.2f}M)")
    print(f"  Worst month bill: {rr['worst_month_bill']/1e6:.2f}M")
    print(f"  Equiv cycles: {rr['equiv_cycles']:.0f}")


--- Replay: M0_I0_R0 (PV=7467, CC=2701, P_B=1710, E_B=12566) ---


  Replay total: 163.11M TWD
  MILP reduced: 84.69M TWD
  Design-to-replay gap: 48.1%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.99M
  Equiv cycles: 0

--- Replay: M1_I0_R0 (PV=7197, CC=2787, P_B=1116, E_B=4652) ---


  Replay total: 154.77M TWD
  MILP reduced: 90.18M TWD
  Design-to-replay gap: 41.7%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.95M
  Equiv cycles: 0

--- Replay: M2_I0_R0 (PV=7719, CC=2938, P_B=1447, E_B=7283) ---


  Replay total: 158.48M TWD
  MILP reduced: 94.10M TWD
  Design-to-replay gap: 40.6%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.87M
  Equiv cycles: 0

--- Replay: M2_I1_R0 (PV=7466, CC=3016, P_B=1362, E_B=7624) ---


  Replay total: 157.59M TWD
  MILP reduced: 94.92M TWD
  Design-to-replay gap: 39.8%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.84M
  Equiv cycles: 0

--- Replay: M2_I1_R1_p3 (PV=7692, CC=3105, P_B=1405, E_B=7866) ---


  Replay total: 158.26M TWD
  MILP reduced: 97.77M TWD
  Design-to-replay gap: 38.2%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.80M
  Equiv cycles: 0

--- Replay: M2_I1_R1_p5 (PV=7842, CC=3165, P_B=1432, E_B=8018) ---


  Replay total: 158.69M TWD
  MILP reduced: 99.67M TWD
  Design-to-replay gap: 37.2%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.77M
  Equiv cycles: 0

--- Replay: M2_I1_R2_p3 (PV=7684, CC=3031, P_B=1346, E_B=7903) ---


  Replay total: 158.45M TWD
  MILP reduced: 95.88M TWD
  Design-to-replay gap: 39.5%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.83M
  Equiv cycles: 0

--- Replay: M2_I1_R2_p5 (PV=7817, CC=3049, P_B=1326, E_B=8002) ---


  Replay total: 158.87M TWD
  MILP reduced: 96.53M TWD
  Design-to-replay gap: 39.2%
  Over-contract months: 9
  RE: 2.1%, T-REC: 3,759,536 kWh (17.41M)
  Worst month bill: 13.82M
  Equiv cycles: 0


In [9]:
# Replay summary table (spec §12.5 metrics)
replay_rows = []
for case_name, rr in replay_results.items():
    milp_row = next((r for r in all_results if r['case'] == case_name), None)
    milp_cost = milp_row['total_cost_M'] * 1e6 if milp_row else 0
    gap = (rr['replay_total'] - milp_cost) / rr['replay_total'] * 100 if rr['replay_total'] > 0 else 0
    replay_rows.append({
        'case': case_name,
        'MILP_M': round(milp_cost / 1e6, 2),
        'replay_M': round(rr['replay_total'] / 1e6, 2),
        'gap_pct': round(gap, 1),
        'over_months': rr['over_contract_months'],
        'worst_bill_M': round(rr['worst_month_bill'] / 1e6, 2),
        'p95_bill_M': round(rr['p95_monthly_bill'] / 1e6, 2),
        'RE_pct': round(rr['re_pct'], 1),
        'TREC_kWh': round(rr['trec_kWh']),
        'equiv_cyc': round(rr['equiv_cycles']),
    })

replay_df = pd.DataFrame(replay_rows)
print(replay_df.to_string(index=False))

# Save
replay_df.to_csv(Path(CFG['output_dir']) / 'replay_summary_main.csv', index=False)
print(f"\nReplay results saved.")

       case  MILP_M  replay_M  gap_pct  over_months  worst_bill_M  p95_bill_M  RE_pct  TREC_kWh  equiv_cyc
   M0_I0_R0   84.69    163.11     48.1            9         13.99       13.14     2.1   3759536          0
   M1_I0_R0   90.18    154.77     41.7            9         13.95       13.10     2.1   3759536          0
   M2_I0_R0   94.10    158.48     40.6            9         13.87       13.03     2.1   3759536          0
   M2_I1_R0   94.92    157.59     39.8            9         13.84       12.99     2.1   3759536          0
M2_I1_R1_p3   97.77    158.26     38.2            9         13.80       12.95     2.1   3759536          0
M2_I1_R1_p5   99.67    158.69     37.2            9         13.77       12.92     2.1   3759536          0
M2_I1_R2_p3   95.88    158.45     39.5            9         13.83       12.99     2.1   3759536          0
M2_I1_R2_p5   96.53    158.87     39.2            9         13.82       12.98     2.1   3759536          0

Replay results saved.
